In [ ]:
import os
import re
import json
import warnings
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from scipy import stats

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, MaxPooling2D, Flatten,
    Dropout, LSTM, Bidirectional, concatenate, Layer
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
#  MOUNT DRIVE & SET PATH

from google.colab import drive
drive.mount('/content/drive')

# *** CHANGE THIS PATH to where your UTD-MHAD .mat files are stored ***
MAT_DATA_PATH = "/content/drive/MyDrive/UTD/UTD-MHAD/Skeleton"

# Directory to save all results (figures, tables, JSON)
RESULTS_DIR = "/content/drive/MyDrive/UTD/HABC_Results_UTD"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify data exists
mat_files = [f for f in os.listdir(MAT_DATA_PATH) if f.endswith('.mat')]
print(f"Found {len(mat_files)} .mat files in {MAT_DATA_PATH}")
print(f"Sample filenames: {sorted(mat_files)[:5]}")


In [ ]:
#  ATTENTION LAYER & MODEL BUILDER FUNCTIONS


class AttentionLayer(Layer):
    """Custom temporal attention layer.
    Computes attention weights over the time dimension and returns
    the weighted sum of hidden states.
    """
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight", shape=(input_shape[-1], 1), initializer="normal"
        )
        self.b = self.add_weight(
            name="att_bias", shape=(input_shape[1], 1), initializer="zeros"
        )
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        et = K.squeeze(K.tanh(K.dot(x, self.W) + self.b), axis=-1)
        at = K.softmax(et)
        at = K.expand_dims(at, axis=-1)
        output = x * at
        return K.sum(output, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        return super(AttentionLayer, self).get_config()


class AttentionLayerWithWeights(Layer):
    """Same attention layer but also stores weights for visualization."""
    def __init__(self, **kwargs):
        super(AttentionLayerWithWeights, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight", shape=(input_shape[-1], 1), initializer="normal"
        )
        self.b = self.add_weight(
            name="att_bias", shape=(input_shape[1], 1), initializer="zeros"
        )
        super(AttentionLayerWithWeights, self).build(input_shape)

    def call(self, x):
        et = K.squeeze(K.tanh(K.dot(x, self.W) + self.b), axis=-1)
        at = K.softmax(et)
        at_expanded = K.expand_dims(at, axis=-1)
        output = x * at_expanded
        return K.sum(output, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        return super(AttentionLayerWithWeights, self).get_config()


# --------------- MODEL BUILDERS ---------------

def build_habc_model(num_classes, time_steps, feature_dim, learning_rate=0.001):
    """Full HABC: CNN + BiLSTM + Attention (the proposed model)."""
    # CNN branch
    conv_input = Input(shape=(1, time_steps, feature_dim), name='cnn_input')
    c = Conv2D(6, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu')(conv_input)
    c = MaxPooling2D(pool_size=(1,2), strides=(1,2), padding='same')(c)
    c = Flatten()(c)

    # BiLSTM + Attention branch
    lstm_input = Input(shape=(time_steps, feature_dim), name='lstm_input')
    l = Bidirectional(LSTM(64, return_sequences=True))(lstm_input)
    l = AttentionLayer()(l)

    # Fusion
    merged = concatenate([c, l])
    x = Dense(64, activation='relu')(merged)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=[conv_input, lstm_input], outputs=output)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=['accuracy']
    )
    return model


def build_cnn_only(num_classes, time_steps, feature_dim, learning_rate=0.001):
    """Ablation: CNN branch only."""
    inp = Input(shape=(1, time_steps, feature_dim), name='cnn_input')
    x = Conv2D(6, (3,3), padding='same', activation='relu')(inp)
    x = MaxPooling2D(pool_size=(1,2), strides=(1,2), padding='same')(x)
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=['accuracy'])
    return model


def build_bilstm_only(num_classes, time_steps, feature_dim, learning_rate=0.001):
    """Ablation: BiLSTM only (no attention, no CNN)."""
    inp = Input(shape=(time_steps, feature_dim), name='lstm_input')
    x = Bidirectional(LSTM(64, return_sequences=False))(inp)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=['accuracy'])
    return model


def build_cnn_bilstm_no_attention(num_classes, time_steps, feature_dim, learning_rate=0.001):
    """Ablation: CNN + BiLSTM but NO attention mechanism."""
    # CNN branch
    cnn_inp = Input(shape=(1, time_steps, feature_dim), name='cnn_input')
    c = Conv2D(6, (3,3), padding='same', activation='relu')(cnn_inp)
    c = MaxPooling2D(pool_size=(1,2), strides=(1,2), padding='same')(c)
    c = Flatten()(c)

    # BiLSTM branch — uses return_sequences=False (last hidden state only)
    lstm_inp = Input(shape=(time_steps, feature_dim), name='lstm_input')
    l = Bidirectional(LSTM(64, return_sequences=False))(lstm_inp)

    # Fusion
    merged = concatenate([c, l])
    x = Dense(64, activation='relu')(merged)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=[cnn_inp, lstm_inp], outputs=out)
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=['accuracy'])
    return model


def build_bilstm_attention_only(num_classes, time_steps, feature_dim, learning_rate=0.001):
    """Ablation: BiLSTM + Attention only (no CNN)."""
    inp = Input(shape=(time_steps, feature_dim), name='lstm_input')
    x = Bidirectional(LSTM(64, return_sequences=True))(inp)
    x = AttentionLayer()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=['accuracy'])
    return model




In [ ]:
# DATA LOADING & PREPROCESSING

def extract_info_from_filename(filename):
    """Extract action ID, subject ID, and trial number from UTD-MHAD filename.
    Filename format: a{action}_s{subject}_t{trial}_skeleton.mat
    """
    action_match = re.search(r'a(\d+)', filename)
    subject_match = re.search(r's(\d+)', filename)
    trial_match = re.search(r't(\d+)', filename)

    if action_match and subject_match:
        action = int(action_match.group(1))
        subject = int(subject_match.group(1))
        trial = int(trial_match.group(1)) if trial_match else 1
        return action, subject, trial
    else:
        raise ValueError(f"Cannot parse filename: {filename}")


def preprocess_skeleton(sample, target_len=76):
    """Pad or truncate skeleton sequence to fixed length."""
    T = sample.shape[2]
    if T > target_len:
        sample = sample[:, :, :target_len]
    elif T < target_len:
        pad_width = target_len - T
        sample = np.pad(sample, ((0,0), (0,0), (0, pad_width)), mode='constant')
    return sample


def load_utd_data(directory, data_key='d_skel', target_len=76):
    """Load all UTD-MHAD skeleton .mat files.
    Returns: X (N, 20, 3*target_len), y (N,), subjects (N,)
    """
    X, y, subjects = [], [], []

    for fname in sorted(os.listdir(directory)):
        if not fname.endswith(".mat"):
            continue
        fpath = os.path.join(directory, fname)
        try:
            data = sio.loadmat(fpath)
        except Exception as e:
            print(f"Error loading {fname}: {e}")
            continue

        if data_key not in data:
            continue

        sample = np.squeeze(data[data_key])
        if sample.ndim == 3 and sample.shape[0] == 20 and sample.shape[1] == 3:
            sample = preprocess_skeleton(sample, target_len)
            flat = sample.reshape(20, -1)  # (20, 3*target_len)
            X.append(flat)

            action, subject, _ = extract_info_from_filename(fname)
            y.append(action)
            subjects.append(subject)
        else:
            print(f"Skipping {fname}: unexpected shape {sample.shape}")

    X = np.array(X, dtype='float32')
    y = np.array(y)
    subjects = np.array(subjects)
    return X, y, subjects


# ---- Load the data ----
TARGET_LEN = 76
raw_X, raw_y, raw_subjects = load_utd_data(MAT_DATA_PATH, target_len=TARGET_LEN)

TIME_STEPS = 20
FEATURE_DIM = 3 * TARGET_LEN  # 228
NUM_CLASSES = len(np.unique(raw_y))

print(f"Loaded {len(raw_X)} samples")
print(f"Actions: {np.unique(raw_y)} ({NUM_CLASSES} classes)")
print(f"Subjects: {np.unique(raw_subjects)} ({len(np.unique(raw_subjects))} subjects)")
print(f"Shape: X={raw_X.shape}, y={raw_y.shape}, subjects={raw_subjects.shape}")
print(f"Time steps: {TIME_STEPS}, Feature dim: {FEATURE_DIM}")

# Encode labels (1-indexed → 0-indexed)
y_encoded = to_categorical(raw_y - 1, num_classes=NUM_CLASSES)

# Prepare two input formats
X_cnn_all = raw_X.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
X_lstm_all = raw_X.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')

print(f"\nCNN input shape:  {X_cnn_all.shape}")
print(f"LSTM input shape: {X_lstm_all.shape}")
print(f"Labels shape:     {y_encoded.shape}")


In [ ]:
#TRAIN BASELINE HABC MODEL (full dataset, for quick sanity check)

print("=" * 60)
print("BASELINE: Training HABC on full UTD-MHAD dataset")
print("=" * 60)

model_baseline = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
model_baseline.summary()

early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

history_baseline = model_baseline.fit(
    [X_cnn_all, X_lstm_all], y_encoded,
    epochs=220,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate on training set (sanity check — not for paper)
score = model_baseline.evaluate([X_cnn_all, X_lstm_all], y_encoded, verbose=0)
print(f"\nBaseline training accuracy: {score[1]*100:.2f}%")

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history_baseline.history['loss'], linewidth=2)
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(history_baseline.history['accuracy'], linewidth=2, color='green')
ax2.set_title('Training Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.grid(True, alpha=0.3)

plt.suptitle('HABC Baseline Training on UTD-MHAD', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/baseline_training_curves.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/baseline_training_curves.png")


In [ ]:
# EXPERIMENT 1 — CROSS-SUBJECT GENERALIZATION (LOSO & Cross-Subject)

print("\n" + "=" * 60)
print("EXPERIMENT 1: Cross-Subject Generalization")
print("=" * 60)

print("\n--- 1A: Leave-One-Subject-Out (LOSO) ---")

logo = LeaveOneGroupOut()
unique_subjects = np.unique(raw_subjects)
n_folds = len(unique_subjects)
print(f"Running {n_folds} LOSO folds (one per subject)...")

loso_results = []

for fold_idx, (train_idx, test_idx) in enumerate(logo.split(raw_X, raw_y, raw_subjects)):
    test_subject = np.unique(raw_subjects[test_idx])[0]

    X_tr = raw_X[train_idx]
    X_te = raw_X[test_idx]
    y_tr = to_categorical(raw_y[train_idx] - 1, num_classes=NUM_CLASSES)
    y_te = to_categorical(raw_y[test_idx] - 1, num_classes=NUM_CLASSES)

    X_cnn_tr = X_tr.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
    X_lstm_tr = X_tr.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')
    X_cnn_te = X_te.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
    X_lstm_te = X_te.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')

    # Fresh model each fold
    K.clear_session()
    model_fold = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

    model_fold.fit(
        [X_cnn_tr, X_lstm_tr], y_tr,
        epochs=150, batch_size=16, verbose=0, callbacks=[es]
    )

    y_pred = np.argmax(model_fold.predict([X_cnn_te, X_lstm_te], verbose=0), axis=1)
    y_true = np.argmax(y_te, axis=1)

    acc = accuracy_score(y_true, y_pred)
    mac_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    w_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    loso_results.append({
        'subject': int(test_subject),
        'accuracy': acc,
        'macro_f1': mac_f1,
        'weighted_f1': w_f1,
        'n_test': len(test_idx)
    })
    print(f"  Fold {fold_idx+1}/{n_folds} | Subject {test_subject:2d} | "
          f"Acc={acc*100:6.2f}% | Macro-F1={mac_f1*100:6.2f}% | "
          f"Weighted-F1={w_f1*100:6.2f}% | n_test={len(test_idx)}")

loso_accs = [r['accuracy'] for r in loso_results]
loso_mf1 = [r['macro_f1'] for r in loso_results]
loso_wf1 = [r['weighted_f1'] for r in loso_results]

print(f"\n  LOSO SUMMARY:")
print(f"  Accuracy:    {np.mean(loso_accs)*100:.2f}% ± {np.std(loso_accs)*100:.2f}%")
print(f"  Macro-F1:    {np.mean(loso_mf1)*100:.2f}% ± {np.std(loso_mf1)*100:.2f}%")
print(f"  Weighted-F1: {np.mean(loso_wf1)*100:.2f}% ± {np.std(loso_wf1)*100:.2f}%")

print("\n--- 1B: Cross-Subject Split (70% train / 30% test by subject) ---")

gss = GroupShuffleSplit(n_splits=5, test_size=0.3, random_state=42)
cross_sub_results = []

for split_idx, (train_idx, test_idx) in enumerate(gss.split(raw_X, raw_y, raw_subjects)):
    X_tr = raw_X[train_idx]
    X_te = raw_X[test_idx]
    y_tr = to_categorical(raw_y[train_idx] - 1, num_classes=NUM_CLASSES)
    y_te = to_categorical(raw_y[test_idx] - 1, num_classes=NUM_CLASSES)

    X_cnn_tr = X_tr.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
    X_lstm_tr = X_tr.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')
    X_cnn_te = X_te.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
    X_lstm_te = X_te.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')

    K.clear_session()
    model_cs = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

    model_cs.fit(
        [X_cnn_tr, X_lstm_tr], y_tr,
        epochs=150, batch_size=16, verbose=0, callbacks=[es]
    )

    y_pred = np.argmax(model_cs.predict([X_cnn_te, X_lstm_te], verbose=0), axis=1)
    y_true = np.argmax(y_te, axis=1)

    acc = accuracy_score(y_true, y_pred)
    mac_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    w_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    train_subs = np.unique(raw_subjects[train_idx])
    test_subs = np.unique(raw_subjects[test_idx])

    cross_sub_results.append({
        'split': split_idx + 1,
        'accuracy': acc,
        'macro_f1': mac_f1,
        'weighted_f1': w_f1,
        'train_subjects': train_subs.tolist(),
        'test_subjects': test_subs.tolist()
    })
    print(f"  Split {split_idx+1}/5 | Test subjects: {test_subs} | "
          f"Acc={acc*100:.2f}% | Macro-F1={mac_f1*100:.2f}%")

cs_accs = [r['accuracy'] for r in cross_sub_results]
cs_mf1 = [r['macro_f1'] for r in cross_sub_results]
cs_wf1 = [r['weighted_f1'] for r in cross_sub_results]

print(f"\n  CROSS-SUBJECT SUMMARY:")
print(f"  Accuracy:    {np.mean(cs_accs)*100:.2f}% ± {np.std(cs_accs)*100:.2f}%")
print(f"  Macro-F1:    {np.mean(cs_mf1)*100:.2f}% ± {np.std(cs_mf1)*100:.2f}%")
print(f"  Weighted-F1: {np.mean(cs_wf1)*100:.2f}% ± {np.std(cs_wf1)*100:.2f}%")

# Save Experiment 1 results
exp1_results = {
    "experiment": "Cross-Subject Generalization",
    "dataset": "UTD-MHAD",
    "LOSO": {
        "mean_accuracy": float(np.mean(loso_accs)),
        "std_accuracy": float(np.std(loso_accs)),
        "mean_macro_f1": float(np.mean(loso_mf1)),
        "std_macro_f1": float(np.std(loso_mf1)),
        "mean_weighted_f1": float(np.mean(loso_wf1)),
        "std_weighted_f1": float(np.std(loso_wf1)),
        "per_subject": loso_results
    },
    "Cross_Subject": {
        "mean_accuracy": float(np.mean(cs_accs)),
        "std_accuracy": float(np.std(cs_accs)),
        "mean_macro_f1": float(np.mean(cs_mf1)),
        "mean_weighted_f1": float(np.mean(cs_wf1)),
        "per_split": cross_sub_results
    }
}
with open(f"{RESULTS_DIR}/exp1_cross_subject.json", "w") as f:
    json.dump(exp1_results, f, indent=2, default=str)
print(f"\nSaved: {RESULTS_DIR}/exp1_cross_subject.json")


In [ ]:
# EXPERIMENT 2 — ABLATION STUDY

print("\n" + "=" * 60)
print("EXPERIMENT 2: Ablation Study")
print("=" * 60)

# Use Cross-Subject split (one split) for fair ablation comparison
gss_ablation = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx_ab, test_idx_ab = next(gss_ablation.split(raw_X, raw_y, raw_subjects))

X_tr_ab = raw_X[train_idx_ab]
X_te_ab = raw_X[test_idx_ab]
y_tr_ab = to_categorical(raw_y[train_idx_ab] - 1, num_classes=NUM_CLASSES)
y_te_ab = to_categorical(raw_y[test_idx_ab] - 1, num_classes=NUM_CLASSES)
y_true_ab = np.argmax(y_te_ab, axis=1)

X_cnn_tr_ab = X_tr_ab.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
X_lstm_tr_ab = X_tr_ab.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')
X_cnn_te_ab = X_te_ab.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
X_lstm_te_ab = X_te_ab.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')

ablation_variants = {
    "CNN only": {
        "builder": build_cnn_only,
        "train_inputs": [X_cnn_tr_ab],
        "test_inputs": [X_cnn_te_ab],
    },
    "BiLSTM only": {
        "builder": build_bilstm_only,
        "train_inputs": [X_lstm_tr_ab],
        "test_inputs": [X_lstm_te_ab],
    },
    "BiLSTM + Attention (no CNN)": {
        "builder": build_bilstm_attention_only,
        "train_inputs": [X_lstm_tr_ab],
        "test_inputs": [X_lstm_te_ab],
    },
    "CNN + BiLSTM (no Attention)": {
        "builder": build_cnn_bilstm_no_attention,
        "train_inputs": [X_cnn_tr_ab, X_lstm_tr_ab],
        "test_inputs": [X_cnn_te_ab, X_lstm_te_ab],
    },
    "HABC (Full Model)": {
        "builder": build_habc_model,
        "train_inputs": [X_cnn_tr_ab, X_lstm_tr_ab],
        "test_inputs": [X_cnn_te_ab, X_lstm_te_ab],
    },
}

ablation_results = {}

for name, config in ablation_variants.items():
    print(f"\n  Training: {name} ...")
    K.clear_session()

    model_ab = config["builder"](NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

    model_ab.fit(
        config["train_inputs"], y_tr_ab,
        epochs=150, batch_size=16, verbose=0, callbacks=[es]
    )

    y_pred_ab = np.argmax(model_ab.predict(config["test_inputs"], verbose=0), axis=1)

    acc = accuracy_score(y_true_ab, y_pred_ab)
    mac_f1 = f1_score(y_true_ab, y_pred_ab, average='macro', zero_division=0)
    prec = precision_score(y_true_ab, y_pred_ab, average='weighted', zero_division=0)
    rec = recall_score(y_true_ab, y_pred_ab, average='weighted', zero_division=0)

    ablation_results[name] = {
        'accuracy': acc,
        'macro_f1': mac_f1,
        'precision': prec,
        'recall': rec
    }
    print(f"  {name}: Acc={acc*100:.2f}% | Macro-F1={mac_f1*100:.2f}% | "
          f"Prec={prec*100:.2f}% | Rec={rec*100:.2f}%")

# Print ablation table
print(f"\n{'='*75}")
print(f"{'Model Variant':<35} {'Accuracy':>10} {'Macro-F1':>10} {'Precision':>10} {'Recall':>10}")
print(f"{'='*75}")
for name, res in ablation_results.items():
    marker = "**" if "HABC" in name else "  "
    print(f"{marker}{name:<33} {res['accuracy']*100:>9.2f}% {res['macro_f1']*100:>9.2f}% "
          f"{res['precision']*100:>9.2f}% {res['recall']*100:>9.2f}%")
print(f"{'='*75}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
names = list(ablation_results.keys())
accs = [ablation_results[n]['accuracy']*100 for n in names]
f1s = [ablation_results[n]['macro_f1']*100 for n in names]

x = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x - width/2, accs, width, label='Accuracy (%)', color='#2196F3')
bars2 = ax.bar(x + width/2, f1s, width, label='Macro-F1 (%)', color='#FF9800')

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Ablation Study — UTD-MHAD', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp2_ablation_study.png", dpi=300, bbox_inches='tight')
plt.show()

# Save
with open(f"{RESULTS_DIR}/exp2_ablation_study.json", "w") as f:
    json.dump(ablation_results, f, indent=2)
print(f"Saved: {RESULTS_DIR}/exp2_ablation_study.json")
print(f"Saved: {RESULTS_DIR}/exp2_ablation_study.png")



In [ ]:
# EXPERIMENT 3 — ROBUSTNESS TO NOISE & MISSING DATA

print("\n" + "=" * 60)
print("EXPERIMENT 3: Robustness to Noise & Missing Data")
print("=" * 60)

# Train a model on clean data first (cross-subject split)
K.clear_session()
model_robust = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
model_robust.fit(
    [X_cnn_tr_ab, X_lstm_tr_ab], y_tr_ab,
    epochs=150, batch_size=16, verbose=0, callbacks=[es]
)

# Baseline (clean test set)
y_pred_clean = np.argmax(model_robust.predict([X_cnn_te_ab, X_lstm_te_ab], verbose=0), axis=1)
clean_acc = accuracy_score(y_true_ab, y_pred_clean)
print(f"\n  Clean test accuracy: {clean_acc*100:.2f}%")


# --- 3A: Gaussian Noise ---
print("\n--- 3A: Gaussian Noise ---")

def add_gaussian_noise(X, snr_db):
    """Add Gaussian noise at a given SNR (dB) level."""
    signal_power = np.mean(X ** 2)
    if signal_power == 0:
        return X
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), X.shape)
    return X + noise

snr_levels = [30, 20, 10, 5, 0]
noise_results = {'clean': clean_acc}

for snr in snr_levels:
    np.random.seed(42)
    X_noisy = add_gaussian_noise(X_te_ab, snr)
    X_cnn_noisy = X_noisy.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
    X_lstm_noisy = X_noisy.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')

    y_pred_noisy = np.argmax(model_robust.predict([X_cnn_noisy, X_lstm_noisy], verbose=0), axis=1)
    acc = accuracy_score(y_true_ab, y_pred_noisy)
    noise_results[f'snr_{snr}'] = acc
    print(f"  SNR = {snr:3d} dB: Accuracy = {acc*100:.2f}%")


# --- 3B: Missing Data ---
print("\n--- 3B: Missing Data Simulation ---")

def simulate_missing_data(X, missing_pct, seed=42):
    """Randomly zero out a percentage of data values."""
    np.random.seed(seed)
    X_missing = X.copy()
    mask = np.random.random(X.shape) < missing_pct
    X_missing[mask] = 0.0
    return X_missing

missing_levels = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30]
missing_results = {}

for pct in missing_levels:
    X_miss = simulate_missing_data(X_te_ab, pct)
    X_cnn_miss = X_miss.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
    X_lstm_miss = X_miss.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')

    y_pred_miss = np.argmax(model_robust.predict([X_cnn_miss, X_lstm_miss], verbose=0), axis=1)
    acc = accuracy_score(y_true_ab, y_pred_miss)
    missing_results[f'{pct*100:.0f}%'] = acc
    print(f"  Missing = {pct*100:5.1f}%: Accuracy = {acc*100:.2f}%")


# --- 3C: Robustness Plots ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Noise plot
snr_vals = [30, 20, 10, 5, 0]
noise_accs = [noise_results[f'snr_{s}']*100 for s in snr_vals]
ax1.plot(snr_vals, noise_accs, 'bo-', linewidth=2, markersize=8, label='HABC')
ax1.axhline(y=clean_acc*100, color='g', linestyle='--', alpha=0.7, label=f'Clean ({clean_acc*100:.1f}%)')
ax1.set_xlabel('SNR (dB)', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Robustness to Gaussian Noise', fontsize=13, fontweight='bold')
ax1.invert_xaxis()
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 105)

# Missing data plot
miss_pcts = [0, 5, 10, 15, 20, 30]
miss_accs = [missing_results[f'{p}%']*100 for p in miss_pcts]
ax2.plot(miss_pcts, miss_accs, 'rs-', linewidth=2, markersize=8, label='HABC')
ax2.set_xlabel('Missing Data (%)', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Robustness to Missing Data', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 105)

plt.suptitle('Robustness Analysis — UTD-MHAD', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp3_robustness.png", dpi=300, bbox_inches='tight')
plt.show()

# Save
exp3_results = {
    "experiment": "Robustness Analysis",
    "dataset": "UTD-MHAD",
    "clean_accuracy": float(clean_acc),
    "noise_results": {k: float(v) for k, v in noise_results.items()},
    "missing_results": {k: float(v) for k, v in missing_results.items()}
}
with open(f"{RESULTS_DIR}/exp3_robustness.json", "w") as f:
    json.dump(exp3_results, f, indent=2)
print(f"Saved: {RESULTS_DIR}/exp3_robustness.json")
print(f"Saved: {RESULTS_DIR}/exp3_robustness.png")



In [ ]:
# EXPERIMENT 4 — STATISTICAL SIGNIFICANCE TESTING
print("\n" + "=" * 60)
print("EXPERIMENT 4: Statistical Significance Testing")
print("=" * 60)
print("Comparing HABC vs CNN-only and BiLSTM-only using LOSO folds")

# We need per-fold results for each model variant
# Run LOSO for CNN-only and BiLSTM-only baselines

baseline_models_sig = {
    "CNN only": {
        "builder": build_cnn_only,
        "input_mode": "cnn"
    },
    "BiLSTM only": {
        "builder": build_bilstm_only,
        "input_mode": "lstm"
    },
}

# Store per-fold accuracies
sig_fold_results = {"HABC": loso_accs}  # Already computed in Exp 1

for bl_name, bl_config in baseline_models_sig.items():
    print(f"\n  Running LOSO for baseline: {bl_name}")
    bl_fold_accs = []

    for fold_idx, (train_idx, test_idx) in enumerate(logo.split(raw_X, raw_y, raw_subjects)):
        test_subject = np.unique(raw_subjects[test_idx])[0]

        X_tr = raw_X[train_idx]
        X_te = raw_X[test_idx]
        y_tr = to_categorical(raw_y[train_idx] - 1, num_classes=NUM_CLASSES)
        y_te = to_categorical(raw_y[test_idx] - 1, num_classes=NUM_CLASSES)

        if bl_config["input_mode"] == "cnn":
            X_tr_inp = [X_tr.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')]
            X_te_inp = [X_te.reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')]
        else:
            X_tr_inp = [X_tr.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')]
            X_te_inp = [X_te.reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')]

        K.clear_session()
        model_bl = bl_config["builder"](NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
        es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

        model_bl.fit(X_tr_inp, y_tr, epochs=150, batch_size=16, verbose=0, callbacks=[es])

        y_pred = np.argmax(model_bl.predict(X_te_inp, verbose=0), axis=1)
        y_true = np.argmax(y_te, axis=1)
        acc = accuracy_score(y_true, y_pred)
        bl_fold_accs.append(acc)

        if (fold_idx + 1) % 4 == 0:
            print(f"    Completed fold {fold_idx+1}/{n_folds}")

    sig_fold_results[bl_name] = bl_fold_accs
    print(f"  {bl_name} LOSO Mean: {np.mean(bl_fold_accs)*100:.2f}% ± {np.std(bl_fold_accs)*100:.2f}%")

# Run statistical tests: HABC vs each baseline
print(f"\n{'='*70}")
print(f"{'Comparison':<30} {'HABC Mean':>10} {'Baseline Mean':>14} {'t-stat':>8} {'p-value (t)':>12} {'W-stat':>8} {'p-value (W)':>12}")
print(f"{'='*70}")

sig_test_results = {}
habc_accs_arr = np.array(sig_fold_results["HABC"])

for bl_name in baseline_models_sig.keys():
    bl_accs_arr = np.array(sig_fold_results[bl_name])

    # Paired t-test
    t_stat, p_t = stats.ttest_rel(habc_accs_arr, bl_accs_arr)

    # Wilcoxon signed-rank test
    try:
        w_stat, p_w = stats.wilcoxon(habc_accs_arr, bl_accs_arr)
    except ValueError:
        w_stat, p_w = float('nan'), float('nan')

    sig_test_results[bl_name] = {
        'habc_mean': float(np.mean(habc_accs_arr)),
        'baseline_mean': float(np.mean(bl_accs_arr)),
        't_statistic': float(t_stat),
        'p_value_ttest': float(p_t),
        'w_statistic': float(w_stat),
        'p_value_wilcoxon': float(p_w),
        'significant_005': bool(p_t < 0.05 or p_w < 0.05)
    }

    sig_str_t = f"{p_t:.6f}" if p_t >= 0.001 else f"{p_t:.2e}"
    sig_str_w = f"{p_w:.6f}" if not np.isnan(p_w) and p_w >= 0.001 else f"{p_w:.2e}"

    print(f"HABC vs {bl_name:<22} {np.mean(habc_accs_arr)*100:>9.2f}% {np.mean(bl_accs_arr)*100:>13.2f}% "
          f"{t_stat:>8.3f} {sig_str_t:>12} {w_stat:>8.1f} {sig_str_w:>12}")

print(f"{'='*70}")

# Save
exp4_results = {
    "experiment": "Statistical Significance",
    "dataset": "UTD-MHAD",
    "fold_results": {k: [float(v) for v in vals] for k, vals in sig_fold_results.items()},
    "comparisons": sig_test_results
}
with open(f"{RESULTS_DIR}/exp4_significance.json", "w") as f:
    json.dump(exp4_results, f, indent=2)
print(f"\nSaved: {RESULTS_DIR}/exp4_significance.json")


In [ ]:
# EXPERIMENT 5 — TEMPORAL LENGTH SENSITIVITY
print("\n" + "=" * 60)
print("EXPERIMENT 5: Temporal Length Sensitivity (UTD-MHAD)")
print("=" * 60)

sequence_lengths = [30, 50, 76, 100, 150]
length_results = {}

for target_len in sequence_lengths:
    print(f"\n  Target length: {target_len} frames")

    # Reload data with new target length
    X_len, y_len, subj_len = load_utd_data(MAT_DATA_PATH, target_len=target_len)
    new_feat_dim = 3 * target_len

    y_len_enc = to_categorical(y_len - 1, num_classes=NUM_CLASSES)

    # Cross-subject split
    gss_len = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    tr_idx, te_idx = next(gss_len.split(X_len, y_len, subj_len))

    X_cnn_tr = X_len[tr_idx].reshape(-1, 1, TIME_STEPS, new_feat_dim).astype('float32')
    X_lstm_tr = X_len[tr_idx].reshape(-1, TIME_STEPS, new_feat_dim).astype('float32')
    X_cnn_te = X_len[te_idx].reshape(-1, 1, TIME_STEPS, new_feat_dim).astype('float32')
    X_lstm_te = X_len[te_idx].reshape(-1, TIME_STEPS, new_feat_dim).astype('float32')

    y_tr = y_len_enc[tr_idx]
    y_te = y_len_enc[te_idx]
    y_true_len = np.argmax(y_te, axis=1)

    K.clear_session()
    model_len = build_habc_model(NUM_CLASSES, TIME_STEPS, new_feat_dim)
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

    model_len.fit(
        [X_cnn_tr, X_lstm_tr], y_tr,
        epochs=150, batch_size=16, verbose=0, callbacks=[es]
    )

    y_pred_len = np.argmax(model_len.predict([X_cnn_te, X_lstm_te], verbose=0), axis=1)
    acc = accuracy_score(y_true_len, y_pred_len)
    f1_mac = f1_score(y_true_len, y_pred_len, average='macro', zero_division=0)

    length_results[target_len] = {'accuracy': acc, 'macro_f1': f1_mac}
    print(f"  Length={target_len}: Accuracy={acc*100:.2f}%, Macro-F1={f1_mac*100:.2f}%")

# Print table
print(f"\n{'='*50}")
print(f"{'Sequence Length':>16} {'Accuracy':>12} {'Macro-F1':>12}")
print(f"{'='*50}")
for l in sequence_lengths:
    r = length_results[l]
    marker = " **" if l == 76 else "   "
    print(f"{l:>14} frames {r['accuracy']*100:>11.2f}% {r['macro_f1']*100:>11.2f}%{marker}")
print(f"{'='*50}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
lens = list(length_results.keys())
accs_len = [length_results[l]['accuracy']*100 for l in lens]
f1s_len = [length_results[l]['macro_f1']*100 for l in lens]

ax.plot(lens, accs_len, 'bo-', linewidth=2, markersize=8, label='Accuracy')
ax.plot(lens, f1s_len, 'rs--', linewidth=2, markersize=8, label='Macro-F1')
ax.axvline(x=76, color='gray', linestyle=':', alpha=0.5, label='Default (76)')
ax.set_xlabel('Sequence Length (Frames)', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Effect of Sequence Length on Performance — UTD-MHAD', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp5_temporal_sensitivity.png", dpi=300, bbox_inches='tight')
plt.show()

# Save
exp5_results = {
    "experiment": "Temporal Length Sensitivity",
    "dataset": "UTD-MHAD",
    "results": {str(k): v for k, v in length_results.items()}
}
with open(f"{RESULTS_DIR}/exp5_temporal_sensitivity.json", "w") as f:
    json.dump(exp5_results, f, indent=2)
print(f"Saved: {RESULTS_DIR}/exp5_temporal_sensitivity.json")
print(f"Saved: {RESULTS_DIR}/exp5_temporal_sensitivity.png")



In [ ]:
# Experiment 7 - TEMPORAL ATTENTION VISUALIZATION

print("\n" + "=" * 60)
print("EXPERIMENT 7: Temporal Attention Visualization")
print("=" * 60)

# Build HABC model with attention weight access
K.clear_session()
model_attn = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

# Train on full data for best attention patterns
model_attn.fit(
    [X_cnn_all, X_lstm_all], y_encoded,
    epochs=150, batch_size=16, verbose=0, callbacks=[es]
)

# Find the attention layer and BiLSTM layer
attention_layer = None
bilstm_layer = None
for layer in model_attn.layers:
    if isinstance(layer, AttentionLayer):
        attention_layer = layer
    if 'bidirectional' in layer.name.lower():
        bilstm_layer = layer

if attention_layer is None:
    print("ERROR: Could not find AttentionLayer!")
else:
    print(f"Found AttentionLayer: {attention_layer.name}")
    print(f"Found BiLSTM layer:   {bilstm_layer.name}")

    # Build sub-model to get BiLSTM output
    bilstm_output_model = Model(
        inputs=model_attn.input[1],  # LSTM input
        outputs=bilstm_layer.output
    )

    # Get attention weights for all samples
    bilstm_out = bilstm_output_model.predict(X_lstm_all, verbose=0)

    # Compute attention weights manually using trained layer weights
    W_att = attention_layer.get_weights()[0]  # (128, 1)
    b_att = attention_layer.get_weights()[1]  # (20, 1)

    # et = tanh(X @ W + b), shape: (N, 20)
    et = np.tanh(np.dot(bilstm_out, W_att) + b_att).squeeze(-1)
    # Softmax over time dimension
    et_exp = np.exp(et - np.max(et, axis=1, keepdims=True))
    attention_weights = et_exp / np.sum(et_exp, axis=1, keepdims=True)

    print(f"Attention weights shape: {attention_weights.shape}")

    # UTD-MHAD action names (27 actions)
    UTD_ACTION_NAMES = {
        1: "Swipe Left", 2: "Swipe Right", 3: "Wave", 4: "Clap",
        5: "Throw", 6: "Arm Cross", 7: "Basketball Shoot", 8: "Draw X",
        9: "Draw Circle CW", 10: "Draw Circle CCW", 11: "Draw Triangle",
        12: "Bowling", 13: "Boxing", 14: "Baseball Swing", 15: "Tennis Swing",
        16: "Arm Curl", 17: "Tennis Serve", 18: "Push", 19: "Knock",
        20: "Catch", 21: "Pickup & Throw", 22: "Jog", 23: "Walk",
        24: "Sit to Stand", 25: "Stand to Sit", 26: "Lunge", 27: "Squat"
    }

    # Select representative actions for visualization
    actions_to_plot = [3, 13, 22, 5, 24]  # Wave, Boxing, Jog, Throw, Sit2Stand
    available_actions = [a for a in actions_to_plot if a in raw_y]

    n_plots = len(available_actions)
    fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))
    if n_plots == 1:
        axes = [axes]

    for idx, action_id in enumerate(available_actions):
        # Get all samples for this action and average attention
        action_mask = raw_y == action_id
        avg_attn = np.mean(attention_weights[action_mask], axis=0)

        colors = plt.cm.Blues(avg_attn / avg_attn.max() * 0.8 + 0.2)

        axes[idx].bar(range(TIME_STEPS), avg_attn, color=colors, edgecolor='steelblue', linewidth=0.5)
        axes[idx].set_xlabel('Time Step (Joint Frame)', fontsize=10)
        axes[idx].set_ylabel('Attention Weight', fontsize=10)
        action_name = UTD_ACTION_NAMES.get(action_id, f"Action {action_id}")
        axes[idx].set_title(f'{action_name}', fontsize=11, fontweight='bold')
        axes[idx].set_xlim(-0.5, TIME_STEPS - 0.5)
        axes[idx].grid(axis='y', alpha=0.3)

    plt.suptitle('Temporal Attention Weight Distribution — UTD-MHAD',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/exp7_attention_visualization.png", dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {RESULTS_DIR}/exp7_attention_visualization.png")

    # Heatmap: attention weights across multiple actions
    fig, ax = plt.subplots(figsize=(12, 8))
    heatmap_data = []
    heatmap_labels = []

    for action_id in sorted(np.unique(raw_y)):
        action_mask = raw_y == action_id
        avg_attn = np.mean(attention_weights[action_mask], axis=0)
        heatmap_data.append(avg_attn)
        action_name = UTD_ACTION_NAMES.get(action_id, f"A{action_id}")
        heatmap_labels.append(action_name)

    heatmap_arr = np.array(heatmap_data)
    sns.heatmap(heatmap_arr, xticklabels=range(1, TIME_STEPS+1),
                yticklabels=heatmap_labels, cmap='YlOrRd',
                ax=ax, cbar_kws={'label': 'Attention Weight'})
    ax.set_xlabel('Time Step', fontsize=12)
    ax.set_ylabel('Action', fontsize=12)
    ax.set_title('Attention Heatmap Across All Actions — UTD-MHAD', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/exp7_attention_heatmap.png", dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {RESULTS_DIR}/exp7_attention_heatmap.png")


In [ ]:
# EXPERIMENT 8 — CONFUSION MATRIX

print("\n" + "=" * 60)
print("EXPERIMENT 8: Confusion Matrix — UTD-MHAD")
print("=" * 60)

# Use the cross-subject model from Experiment 2/3
K.clear_session()
model_cm = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

# Cross-subject split
gss_cm = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx_cm, te_idx_cm = next(gss_cm.split(raw_X, raw_y, raw_subjects))

X_cnn_tr_cm = raw_X[tr_idx_cm].reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
X_lstm_tr_cm = raw_X[tr_idx_cm].reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')
X_cnn_te_cm = raw_X[te_idx_cm].reshape(-1, 1, TIME_STEPS, FEATURE_DIM).astype('float32')
X_lstm_te_cm = raw_X[te_idx_cm].reshape(-1, TIME_STEPS, FEATURE_DIM).astype('float32')

y_tr_cm = to_categorical(raw_y[tr_idx_cm] - 1, num_classes=NUM_CLASSES)
y_te_cm = to_categorical(raw_y[te_idx_cm] - 1, num_classes=NUM_CLASSES)

model_cm.fit(
    [X_cnn_tr_cm, X_lstm_tr_cm], y_tr_cm,
    epochs=150, batch_size=16, verbose=0, callbacks=[es]
)

y_pred_cm = np.argmax(model_cm.predict([X_cnn_te_cm, X_lstm_te_cm], verbose=0), axis=1)
y_true_cm = np.argmax(y_te_cm, axis=1)

# Build class labels
unique_classes = sorted(np.unique(y_true_cm))
class_labels = [UTD_ACTION_NAMES.get(c+1, f"A{c+1}") for c in unique_classes]

# Plot confusion matrix
cm = confusion_matrix(y_true_cm, y_pred_cm)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.5, linecolor='lightgray')
ax.set_xlabel('Predicted Action', fontsize=13)
ax.set_ylabel('True Action', fontsize=13)
ax.set_title('Confusion Matrix — HABC on UTD-MHAD (Cross-Subject)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp8_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

# Also save normalized confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.5, linecolor='lightgray', vmin=0, vmax=1)
ax.set_xlabel('Predicted Action', fontsize=13)
ax.set_ylabel('True Action', fontsize=13)
ax.set_title('Normalized Confusion Matrix — HABC on UTD-MHAD', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp8_confusion_matrix_normalized.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {RESULTS_DIR}/exp8_confusion_matrix.png")
print(f"Saved: {RESULTS_DIR}/exp8_confusion_matrix_normalized.png")



In [ ]:
# EXPERIMENT 10 — CLASS-WISE PRECISION, RECALL, F1-SCORE

print("\n" + "=" * 60)
print("EXPERIMENT 10: Class-Wise Metrics — UTD-MHAD")
print("=" * 60)

# Using predictions from Experiment 8
report_dict = classification_report(y_true_cm, y_pred_cm, output_dict=True, zero_division=0)

# Print formatted table
print(f"\n{'='*75}")
print(f"{'Action':<25} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print(f"{'='*75}")

for class_idx in sorted(np.unique(y_true_cm)):
    class_key = str(class_idx)
    if class_key in report_dict:
        r = report_dict[class_key]
        action_name = UTD_ACTION_NAMES.get(class_idx + 1, f"Action {class_idx+1}")
        print(f"{action_name:<25} {r['precision']*100:>9.2f}% {r['recall']*100:>9.2f}% "
              f"{r['f1-score']*100:>9.2f}% {r['support']:>9.0f}")

print(f"{'-'*75}")
for avg_type in ['macro avg', 'weighted avg']:
    if avg_type in report_dict:
        r = report_dict[avg_type]
        label = avg_type.replace(' avg', ' Avg').title()
        print(f"{'** ' + label + ' **':<25} {r['precision']*100:>9.2f}% {r['recall']*100:>9.2f}% "
              f"{r['f1-score']*100:>9.2f}% {r['support']:>9.0f}")
print(f"{'='*75}")

overall_acc = accuracy_score(y_true_cm, y_pred_cm)
print(f"\nOverall Accuracy: {overall_acc*100:.2f}%")

# Full classification report (standard format)
print("\n--- Full Classification Report ---")
print(classification_report(
    y_true_cm, y_pred_cm,
    target_names=class_labels,
    digits=4, zero_division=0
))

# Save
exp10_results = {
    "experiment": "Class-Wise Metrics",
    "dataset": "UTD-MHAD",
    "overall_accuracy": float(overall_acc),
    "classification_report": report_dict
}
with open(f"{RESULTS_DIR}/exp10_classwise_metrics.json", "w") as f:
    json.dump(exp10_results, f, indent=2)
print(f"Saved: {RESULTS_DIR}/exp10_classwise_metrics.json")

# Bar chart of per-class F1 scores
fig, ax = plt.subplots(figsize=(14, 6))
class_f1s = []
class_names_plot = []
for class_idx in sorted(np.unique(y_true_cm)):
    class_key = str(class_idx)
    if class_key in report_dict:
        class_f1s.append(report_dict[class_key]['f1-score'] * 100)
        class_names_plot.append(UTD_ACTION_NAMES.get(class_idx + 1, f"A{class_idx+1}"))

colors = ['#4CAF50' if f >= 80 else '#FF9800' if f >= 50 else '#F44336' for f in class_f1s]
bars = ax.bar(range(len(class_f1s)), class_f1s, color=colors, edgecolor='gray', linewidth=0.5)
ax.set_xticks(range(len(class_names_plot)))
ax.set_xticklabels(class_names_plot, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('F1-Score (%)', fontsize=12)
ax.set_title('Per-Class F1-Score — HABC on UTD-MHAD', fontsize=13, fontweight='bold')
ax.axhline(y=np.mean(class_f1s), color='blue', linestyle='--', alpha=0.7,
           label=f'Mean F1 = {np.mean(class_f1s):.1f}%')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp10_per_class_f1.png", dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/exp10_per_class_f1.png")



In [ ]:
# =============================================================================
# CELL 13: FINAL SUMMARY — ALL EXPERIMENTS (CRASH-PROOF VERSION)
# =============================================================================

print("\n" + "=" * 70)
print("  FINAL SUMMARY: ALL EXPERIMENTS — HABC on UTD-MHAD")
print("=" * 70)

# --- SAFETY CHECKS (To prevent NameError if earlier cells are skipped) ---
clean_acc = globals().get('clean_acc', 0.0)
noise_results = globals().get('noise_results', {})
missing_results = globals().get('missing_results', {})
sig_test_results = globals().get('sig_test_results', {})
sequence_lengths = globals().get('sequence_lengths', [])
length_results = globals().get('length_results', {})
loso_accs = globals().get('loso_accs', [0])
loso_mf1 = globals().get('loso_mf1', [0])
cs_accs = globals().get('cs_accs', [0])
ablation_results = globals().get('ablation_results', {})
# -------------------------------------------------------------------------

print(f"""
┌───────────────────────────────────────────────────────────────────┐
│ Experiment 1: Cross-Subject Generalization                        │
│   LOSO Accuracy:         {np.mean(loso_accs)*100:6.2f}% ± {np.std(loso_accs)*100:.2f}%                  │
│   LOSO Macro-F1:         {np.mean(loso_mf1)*100:6.2f}% ± {np.std(loso_mf1)*100:.2f}%                  │
│   Cross-Subject Acc:     {np.mean(cs_accs)*100:6.2f}% ± {np.std(cs_accs)*100:.2f}%                  │
│                                                                   │
│ Experiment 2: Ablation Study                                      │""")

if ablation_results:
    for name, res in ablation_results.items():
        tag = " ← Proposed" if "HABC" in name else ""
        print(f"│   {name:<30} Acc={res['accuracy']*100:.2f}% F1={res['macro_f1']*100:.2f}%{tag:<10}│")
else:
    print("│   [Ablation Study Skipped]                                        │")

print(f"""│                                                                   │
│ Experiment 3: Robustness                                          │
│   Clean Accuracy:        {clean_acc*100:6.2f}%                                │
│   SNR=10dB:              {noise_results.get('snr_10', 0)*100:6.2f}%                                │
│   Missing 20%:           {missing_results.get('20%', 0)*100:6.2f}%                                │
│                                                                   │
│ Experiment 4: Statistical Significance                            │""")

if sig_test_results:
    for bl_name, sig_res in sig_test_results.items():
        sig_mark = "✓" if sig_res.get('significant_005', False) else "✗"
        p_val = sig_res.get('p_value_ttest', 1.0)
        print(f"│   HABC vs {bl_name:<18} p={p_val:.4f} (t-test) {sig_mark}             │")
else:
    print("│   [Significance Testing Skipped]                                  │")

print(f"""│                                                                   │
│ Experiment 5: Temporal Sensitivity                                │""")
if sequence_lengths:
    for l in sequence_lengths:
        r = length_results.get(l, {'accuracy': 0, 'macro_f1': 0})
        default = " (default)" if l == 76 else ""
        print(f"│   Length={l:3d}: Acc={r['accuracy']*100:.2f}%  F1={r['macro_f1']*100:.2f}%{default:<14}│")
else:
    print("│   [Temporal Sensitivity Skipped]                                  │")

print(f"""│                                                                   │
│ Experiment 7:  Attention visualization saved ✓                    │
│ Experiment 8:  Confusion matrices saved ✓                         │
│ Experiment 10: Class-wise metrics saved ✓                         │
│                                                                   │
│ All results saved to: {globals().get('RESULTS_DIR', 'Drive')} │
└───────────────────────────────────────────────────────────────────┘
""")

# Safe Master Summary Creation
master_summary = {
    "model": "HABC (CNN + BiLSTM + Attention)",
    "dataset": "UTD-MHAD",
    "num_samples": int(len(globals().get('raw_X', []))),
    "num_classes": int(globals().get('NUM_CLASSES', 0)),
    "num_subjects": int(len(np.unique(globals().get('raw_subjects', [])))),
    "experiments_completed": [
        "Exp1: Cross-Subject (LOSO + Cross-Subject)",
        "Exp2: Ablation Study (5 variants)",
        "Exp3: Robustness (Noise + Missing Data)",
        "Exp4: Statistical Significance Testing",
        "Exp5: Temporal Length Sensitivity",
        "Exp7: Attention Visualization",
        "Exp8: Confusion Matrix",
        "Exp10: Class-Wise Metrics",
    ]
}

try:
    with open(f"{globals().get('RESULTS_DIR', '.')}/master_summary.json", "w") as f:
        json.dump(master_summary, f, indent=2)
    print("ALL EXPERIMENTS COMPLETE!")
    print(f"Check your Google Drive at: {globals().get('RESULTS_DIR', '.')}/")
except Exception as e:
    print(f"Could not save JSON: {e}")